# 1. Imports

In [46]:
import os
import json
import random
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
     roc_curve
)

import matplotlib.pyplot as plt

from sklearn.cluster import KMeans, SpectralClustering
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE


In [47]:
# Environment setup
# Running in Colab with Google Drive (default):
BASE_DIR = Path("/content/gdrive/MyDrive/THESIS/thesis_xai_healthcare")
DATA_DIR = BASE_DIR / "data"

# Running locally (PyCharm, Jupyter, VSCode):
# BASE_DIR = Path(".")
# DATA_DIR = BASE_DIR / "data"

# Colab only: mount Google Drive
# Skip this cell if running locally
from google.colab import drive
drive.mount("/content/gdrive")

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


# 2. Dataset Upload

In [48]:
# Colab only: upload dataset files
# Skip this cell if running locally — place dataset files in the working directory instead
from google.colab import files
files.upload()

Saving processed.cleveland.data to processed.cleveland (3).data


{'processed.cleveland (3).data': b'63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0\n67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2\n67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1\n37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0\n41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0\n56.0,1.0,2.0,120.0,236.0,0.0,0.0,178.0,0.0,0.8,1.0,0.0,3.0,0\n62.0,0.0,4.0,140.0,268.0,0.0,2.0,160.0,0.0,3.6,3.0,2.0,3.0,3\n57.0,0.0,4.0,120.0,354.0,0.0,0.0,163.0,1.0,0.6,1.0,0.0,3.0,0\n63.0,1.0,4.0,130.0,254.0,0.0,2.0,147.0,0.0,1.4,2.0,1.0,7.0,2\n53.0,1.0,4.0,140.0,203.0,1.0,2.0,155.0,1.0,3.1,3.0,0.0,7.0,1\n57.0,1.0,4.0,140.0,192.0,0.0,0.0,148.0,0.0,0.4,2.0,0.0,6.0,0\n56.0,0.0,2.0,140.0,294.0,0.0,2.0,153.0,0.0,1.3,2.0,0.0,3.0,0\n56.0,1.0,3.0,130.0,256.0,1.0,2.0,142.0,1.0,0.6,2.0,1.0,6.0,2\n44.0,1.0,2.0,120.0,263.0,0.0,0.0,173.0,0.0,0.0,1.0,0.0,7.0,0\n52.0,1.0,3.0,172.0,199.0,1.0,0.0,162.0,0.0,0.5,1.0,0.0,7.0,0\n57.0,1.0,3.0,150.0,168.0,0.0,0.0,17

# 3. Dataset Configurations

In [49]:
WDBC_BASE_FEATURES = [
    "radius", "texture", "perimeter", "area", "smoothness",
    "compactness", "concavity", "concave_points", "symmetry",
    "fractal_dimension"
]

WDBC_FEATURES = (
    [f"mean_{feature}" for feature in WDBC_BASE_FEATURES]
    + [f"se_{feature}" for feature in WDBC_BASE_FEATURES]
    + [f"worst_{feature}" for feature in WDBC_BASE_FEATURES]
)

DATASETS = {
    "heart_disease": {
        "file_path": "processed.cleveland.data",
        "read_csv_kwargs": {
            "names": [
                "age", "sex", "cp", "trestbps", "chol", "fbs",
                "restecg", "thalach", "exang", "oldpeak",
                "slope", "ca", "thal", "target"
            ],
            "na_values": "?"
        },
        "target_column": "target",
        "id_column": None,
        "numeric_features": ["age", "trestbps", "chol", "thalach", "oldpeak"],
        "categorical_features": ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"],
        "target_type": "heart_binary",
        "zero_as_missing": []
    },

    "diabetes": {
        "file_path": "diabetes.csv",
        "read_csv_kwargs": {},
        "target_column": "Outcome",
        "id_column": None,
        "numeric_features": [
            "Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
            "Insulin", "BMI", "DiabetesPedigreeFunction", "Age"
        ],
        "categorical_features": [],
        "target_type": "already_binary",
        "zero_as_missing": [
            "Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"
        ]
    },

    "breast_cancer_wdbc": {
        "file_path": "wdbc.data",
        "read_csv_kwargs": {
            "names": ["id", "diagnosis"] + WDBC_FEATURES
        },
        "target_column": "diagnosis",
        "id_column": "id",
        "numeric_features": WDBC_FEATURES,
        "categorical_features": [],
        "target_type": "wdbc_diagnosis",
        "zero_as_missing": []
    }
}

# 4. Select Dataset

In [50]:
dataset_name = "heart_disease"
# dataset_name = "diabetes"
# dataset_name = "breast_cancer_wdbc"

config = DATASETS[dataset_name]
DATA_FILE_PATH = DATA_DIR / config["file_path"]

print("Selected dataset:", dataset_name)
print("Expected file:", DATA_FILE_PATH)

Selected dataset: heart_disease
Expected file: /content/gdrive/MyDrive/THESIS/thesis_xai_healthcare/data/processed.cleveland.data


# 5. Load Dataset

In [51]:
if not DATA_FILE_PATH.exists():
    raise FileNotFoundError(
        f"File not found: {DATA_FILE_PATH}. "
        "Please place the dataset file in the data/ folder."
    )

df = pd.read_csv(
    DATA_FILE_PATH,
    **config["read_csv_kwargs"]
)

print(f"Dataset loaded successfully: {dataset_name}")
print("Shape:", df.shape)
print(df.head())
print("\nMissing values:")
print(df.isna().sum())

Dataset loaded successfully: heart_disease
Shape: (303, 14)
    age  sex   cp  trestbps   chol  fbs  restecg  thalach  exang  oldpeak  \
0  63.0  1.0  1.0     145.0  233.0  1.0      2.0    150.0    0.0      2.3   
1  67.0  1.0  4.0     160.0  286.0  0.0      2.0    108.0    1.0      1.5   
2  67.0  1.0  4.0     120.0  229.0  0.0      2.0    129.0    1.0      2.6   
3  37.0  1.0  3.0     130.0  250.0  0.0      0.0    187.0    0.0      3.5   
4  41.0  0.0  2.0     130.0  204.0  0.0      2.0    172.0    0.0      1.4   

   slope   ca  thal  target  
0    3.0  0.0   6.0       0  
1    2.0  3.0   3.0       2  
2    2.0  2.0   7.0       1  
3    3.0  0.0   3.0       0  
4    1.0  0.0   3.0       0  

Missing values:
age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          4
thal        2
target      0
dtype: int64


## Feature Summary

**Heart Disease Cleveland**  
Clinical and exercise-test variables including age, sex, chest pain type, resting blood pressure, cholesterol, fasting blood sugar, ECG results, maximum heart rate, exercise-induced angina, ST depression, ST slope, fluoroscopy vessels, and thalassemia result.

**Pima Indians Diabetes**  
Diagnostic variables including pregnancies, glucose, blood pressure, skin thickness, insulin, BMI, diabetes pedigree function, and age. Clinically impossible zero values in selected columns are treated as missing.

**Breast Cancer WDBC**  
Thirty numeric features computed from digitized images of breast mass fine needle aspirates. Ten morphology characteristics are measured as mean, standard error, and worst value.


# 6. Data Preparation


In [52]:
target_column = config["target_column"]
id_column = config["id_column"]

# Keep sample IDs for later saving predictions/clusters
if id_column is not None:
    sample_ids = df[id_column].copy()
else:
    sample_ids = pd.Series(range(len(df)), name="sample_id")

# Separate target first, before converting feature columns
target_raw = df[target_column].copy()

# Drop target and ID column from features
columns_to_drop = [target_column]

if id_column is not None:
    columns_to_drop.append(id_column)

X = df.drop(columns=columns_to_drop)

# Convert feature columns to numeric
X = X.apply(pd.to_numeric)

# Treat clinically impossible zeros as missing for selected datasets
for column in config["zero_as_missing"]:
    X[column] = X[column].replace(0, np.nan)

# Encode target based on dataset type
if config["target_type"] == "heart_binary":
    y = target_raw.apply(lambda value: 0 if value == 0 else 1)

elif config["target_type"] == "already_binary":
    y = pd.to_numeric(target_raw)

elif config["target_type"] == "wdbc_diagnosis":
    y = target_raw.map({"B": 0, "M": 1})

else:
    raise ValueError("Unknown target type:", config["target_type"])

print("Features shape:", X.shape)
print("Target distribution:")
print(y.value_counts())
print("\nFeature preview:")
print(X.head())

Features shape: (303, 13)
Target distribution:
target
0    164
1    139
Name: count, dtype: int64

Feature preview:
    age  sex   cp  trestbps   chol  fbs  restecg  thalach  exang  oldpeak  \
0  63.0  1.0  1.0     145.0  233.0  1.0      2.0    150.0    0.0      2.3   
1  67.0  1.0  4.0     160.0  286.0  0.0      2.0    108.0    1.0      1.5   
2  67.0  1.0  4.0     120.0  229.0  0.0      2.0    129.0    1.0      2.6   
3  37.0  1.0  3.0     130.0  250.0  0.0      0.0    187.0    0.0      3.5   
4  41.0  0.0  2.0     130.0  204.0  0.0      2.0    172.0    0.0      1.4   

   slope   ca  thal  
0    3.0  0.0   6.0  
1    2.0  3.0   3.0  
2    2.0  2.0   7.0  
3    3.0  0.0   3.0  
4    1.0  0.0   3.0  


# 7. Feature Definition and Train-Test Split

In [53]:
numeric_features = config["numeric_features"]
categorical_features = config["categorical_features"]

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

# Split BEFORE imputation/scaling to avoid data leakage
X_train, X_test, y_train, y_test, sample_ids_train, sample_ids_test = train_test_split(
    X,
    y,
    sample_ids,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTest target distribution:")
print(y_test.value_counts())

Numeric features: ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
Categorical features: ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
Training set shape: (242, 13)
Test set shape: (61, 13)

Training target distribution:
target
0    131
1    111
Name: count, dtype: int64

Test target distribution:
target
0    33
1    28
Name: count, dtype: int64


# 8. Preprocessing Pipeline

In [54]:
# Numeric columns:
# median imputation + scaling
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical columns:
# most frequent imputation + one-hot encoding
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing branches
transformers = [
    ("num", numeric_transformer, numeric_features)
]

if len(categorical_features) > 0:
    transformers.append(("cat", categorical_transformer, categorical_features))

preprocessor = ColumnTransformer(transformers=transformers)

# Fit preprocessing only on training data
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Convert sparse matrices to dense arrays if needed
if hasattr(X_train_processed, "toarray"):
    X_train_processed = X_train_processed.toarray()

if hasattr(X_test_processed, "toarray"):
    X_test_processed = X_test_processed.toarray()

# Save processed feature names for later explainability
feature_names = preprocessor.get_feature_names_out()

print("Processed training shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)
print("Number of processed features:", len(feature_names))
print(feature_names)

Processed training shape: (242, 28)
Processed test shape: (61, 28)
Number of processed features: 28
['num__age' 'num__trestbps' 'num__chol' 'num__thalach' 'num__oldpeak'
 'cat__sex_0.0' 'cat__sex_1.0' 'cat__cp_1.0' 'cat__cp_2.0' 'cat__cp_3.0'
 'cat__cp_4.0' 'cat__fbs_0.0' 'cat__fbs_1.0' 'cat__restecg_0.0'
 'cat__restecg_1.0' 'cat__restecg_2.0' 'cat__exang_0.0' 'cat__exang_1.0'
 'cat__slope_1.0' 'cat__slope_2.0' 'cat__slope_3.0' 'cat__ca_0.0'
 'cat__ca_1.0' 'cat__ca_2.0' 'cat__ca_3.0' 'cat__thal_3.0' 'cat__thal_6.0'
 'cat__thal_7.0']


# 9. Conversion to PyTorch Tensors

In [55]:
X_train_tensor = torch.tensor(X_train_processed, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_processed, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.float32).view(-1, 1)
y_test_tensor = torch.tensor(y_test.to_numpy(), dtype=torch.float32).view(-1, 1)

print("X_train_tensor:", X_train_tensor.shape)
print("y_train_tensor:", y_train_tensor.shape)
print("X_test_tensor:", X_test_tensor.shape)
print("y_test_tensor:", y_test_tensor.shape)

X_train_tensor: torch.Size([242, 28])
y_train_tensor: torch.Size([242, 1])
X_test_tensor: torch.Size([61, 28])
y_test_tensor: torch.Size([61, 1])


# 10. Neural Network Architecture

In [56]:
class TabularMLP(nn.Module):
    """
    MLP for binary classification on tabular healthcare datasets.
    Also returns hidden-layer activations for Phase 2 clustering.
    """
    def __init__(self, input_size):
        super(TabularMLP, self).__init__()

        self.hidden1 = nn.Linear(input_size, 32)
        self.relu1 = nn.ReLU()

        self.hidden2 = nn.Linear(32, 16)
        self.relu2 = nn.ReLU()

        self.output = nn.Linear(16, 1)

    def forward(self, x, return_activations=False):
        layer1 = self.relu1(self.hidden1(x))
        layer2 = self.relu2(self.hidden2(layer1))
        logits = self.output(layer2)

        if return_activations:
            return logits, {
                "layer1": layer1,
                "layer2": layer2
            }

        return logits

# 11. Model Initialization

In [57]:
# Random seeds before model initialization for reproducible weights
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

input_size = X_train_tensor.shape[1]
model = TabularMLP(input_size)

print("Dataset:", dataset_name)
print("Input size:", input_size)
print(model)

Dataset: heart_disease
Input size: 28
TabularMLP(
  (hidden1): Linear(in_features=28, out_features=32, bias=True)
  (relu1): ReLU()
  (hidden2): Linear(in_features=32, out_features=16, bias=True)
  (relu2): ReLU()
  (output): Linear(in_features=16, out_features=1, bias=True)
)


# 12. Loss Function and Optimizer

In [58]:
# Binary classification loss function
# Define loss and optimizer

criterion = nn.BCEWithLogitsLoss()

# Adam optimizer
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

# 13. Model Training

In [59]:
num_epochs = 200

print("Training model for dataset:", dataset_name)

for epoch in range(num_epochs):
    model.train()

    # Forward pass
    outputs = model(X_train_tensor)

    # Calculate loss
    loss = criterion(outputs, y_train_tensor)

    # Backward pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Print progress every 20 epochs
    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")


Training model for dataset: heart_disease
Epoch [20/200], Loss: 0.6657
Epoch [40/200], Loss: 0.5935
Epoch [60/200], Loss: 0.4736
Epoch [80/200], Loss: 0.3788
Epoch [100/200], Loss: 0.3348
Epoch [120/200], Loss: 0.3123
Epoch [140/200], Loss: 0.2947
Epoch [160/200], Loss: 0.2796
Epoch [180/200], Loss: 0.2641
Epoch [200/200], Loss: 0.2477


# 14. Model Evaluation

In [60]:
model.eval()

with torch.no_grad():
    # Training set predictions
    train_logits = model(X_train_tensor)
    train_probabilities = torch.sigmoid(train_logits)
    train_predictions = (train_probabilities >= 0.5).int()

    # Test set predictions
    test_logits = model(X_test_tensor)
    test_probabilities = torch.sigmoid(test_logits)
    test_predictions = (test_probabilities >= 0.5).int()

# Convert tensors to NumPy arrays for sklearn metrics
y_train_pred = train_predictions.numpy().ravel()
y_test_pred = test_predictions.numpy().ravel()
y_test_proba = test_probabilities.numpy().ravel()

print("Dataset:", dataset_name)

print("\nTraining Accuracy:")
print(accuracy_score(y_train, y_train_pred))

print("\nTest Accuracy:")
print(accuracy_score(y_test, y_test_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_test_pred))

print("\nROC-AUC:")
print(roc_auc_score(y_test, y_test_proba))

Dataset: heart_disease

Training Accuracy:
0.9173553719008265

Test Accuracy:
0.9016393442622951

Confusion Matrix:
[[29  4]
 [ 2 26]]

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.88      0.91        33
           1       0.87      0.93      0.90        28

    accuracy                           0.90        61
   macro avg       0.90      0.90      0.90        61
weighted avg       0.90      0.90      0.90        61


ROC-AUC:
0.9621212121212122


# 15. Save Phase 1 Artifacts for Later Phases

In [61]:
PHASE1_DIR = BASE_DIR / "artifacts" / dataset_name / "phase1_model"
PHASE1_DIR.mkdir(parents=True, exist_ok=True)

X_train_np = np.asarray(X_train_processed).astype(np.float32)
X_test_np = np.asarray(X_test_processed).astype(np.float32)

y_train_np = np.asarray(y_train).reshape(-1, 1).astype(np.float32)
y_test_np = np.asarray(y_test).reshape(-1, 1).astype(np.float32)

torch.save(model.state_dict(), PHASE1_DIR / "pytorch_model_state_dict.pt")
joblib.dump(preprocessor, PHASE1_DIR / "preprocessor.joblib")

np.save(PHASE1_DIR / "X_train_np.npy", X_train_np)
np.save(PHASE1_DIR / "X_test_np.npy", X_test_np)
np.save(PHASE1_DIR / "y_train_np.npy", y_train_np)
np.save(PHASE1_DIR / "y_test_np.npy", y_test_np)

np.save(PHASE1_DIR / "sample_ids_train.npy", np.asarray(sample_ids_train))
np.save(PHASE1_DIR / "sample_ids_test.npy", np.asarray(sample_ids_test))

model_config = {
    "dataset_name": dataset_name,
    "original_file_name": config["file_path"],
    "input_size": int(X_train_np.shape[1]),
    "hidden1_size": 32,
    "hidden2_size": 16,
    "output_size": 1,
    "loss_function": "BCEWithLogitsLoss",
    "optimizer": "Adam",
    "final_activation": "linear_logit"
}

with open(PHASE1_DIR / "model_config.json", "w") as f:
    json.dump(model_config, f, indent=4)

print("Saved Phase 1 artifacts to:")
print(PHASE1_DIR)
print("Files:")
print(os.listdir(PHASE1_DIR))
print("Standardized Phase 1 artifact arrays:")
print("X_train_np:", X_train_np.shape)
print("X_test_np:", X_test_np.shape)
print("y_train_np:", y_train_np.shape)
print("y_test_np:", y_test_np.shape)

Saved Phase 1 artifacts to:
/content/gdrive/MyDrive/THESIS/thesis_xai_healthcare/artifacts/heart_disease/phase1_model
Files:
['pytorch_model_state_dict.pt', 'X_train_np.npy', 'preprocessor.joblib', 'y_train_np.npy', 'X_test_np.npy', 'y_test_np.npy', 'sample_ids_train.npy', 'sample_ids_test.npy', 'model_config.json']
Standardized Phase 1 artifact arrays:
X_train_np: (242, 28)
X_test_np: (61, 28)
y_train_np: (242, 1)
y_test_np: (61, 1)
